# 📚 Demo Sprint 1 — Primer Incremento Funcional
## Sistema de Gestión y Compra de Libros en Línea

**Asignatura:** IS873 — Laboratorio de Software (Semestre 2026-2)  
**Universidad Tecnológica de Pereira (UTP)** — Facultad de Ingenierías  
**Evaluación:** Nota 2 — Sprint Review 1 (20% de la calificación final)  

### 👥 Equipo de Desarrollo:
* **Jose David Marín Giraldo** (`josedavid.marin@utp.edu.co`)
* **Edwin Osorio Cartagena**
* **Maicol Londoño Hernández**

---
### 🎯 Sprint Goal
> *"Construir y desplegar el núcleo operativo de seguridad y control de acceso del sistema mediante autenticación JWT, autoservicio de clientes y control de acceso basado en roles (RBAC), garantizando de forma atómica la restricción de compras para cuentas administrativas (RF-031) en una arquitectura cliente-servidor funcional y verificable."*

---
### 🔍 Qué demuestra este incremento y qué NO demuestra todavía

#### ✅ Qué Demuestra Este Incremento (Sprint 1 — 21 Story Points):
1. **HU-01 (5 SP / RF-027):** Registro de nuevos clientes con DNI y correo único, validación de formato y almacenamiento con sal y hashing `bcrypt`.
2. **HU-04 (3 SP / RF-026, RF-028):** Inicio de sesión con emisión de token Bearer JWT (`exp` 24h, claims tipados de rol e ID de usuario).
3. **HU-05 (1 SP):** Cierre de sesión y desvinculación local de credenciales.
4. **HU-06 (2 SP / RF-028):** Cambio seguro de contraseña exigiendo la verificación obligatoria de la contraseña anterior antes de rehashear.
5. **HU-07 (3 SP / RF-021, RF-028):** Consulta de perfil autenticado (`/me`), monedero virtual ($150.000 COP) y actualización de teléfono y dirección de correspondencia.
6. **HU-02 (5 SP / RF-030):** Creación de personal con rol `ADMINISTRADOR` por parte exclusiva de `ROOT`, con auditoría y bloqueo `HTTP 403 Forbidden` a clientes.
7. **HU-03 (2 SP / RF-031):** Regla de Negocio RF-031: Prohibición absoluta de compras o reservas personales a usuarios `ADMINISTRADOR` y `ROOT` (`HTTP 403`), autorizando exclusivamente a cuentas `CLIENTE` (`HTTP 200`).

#### ❌ Qué NO Demuestra Todavía:
* **No demuestra catálogo bibliográfico ni CRUD de inventario:** La administración de títulos y copias físicas (`EJEMPLAR`) corresponde al **Sprint 2 (27 SP)**.
* **No demuestra el worker de reservas de 24h:** La caducidad asíncrona de reservas fue validada en la PoC de la Nota 1 y se integrará en el **Sprint 3**.
* **No demuestra integración con pasarela financiera externa ni Google Maps:** Planificados para los Sprints 3 y 4.

## ⚙️ 0. Configuración del Cliente de Pruebas y Estado del API
Verificamos la conectividad con el backend FastAPI (por defecto en `http://localhost:8000`).

In [ ]:
import requests
import json
import random

BASE_URL = "http://localhost:8000/api/v1"
ROOT_API = "http://localhost:8000"

try:
    res = requests.get(ROOT_API + "/")
    print("[✓] Conexión establecida exitosamente con el Backend FastAPI:")
    print(json.dumps(res.json(), indent=2, ensure_ascii=False))
except Exception as e:
    print("[!] El backend no parece estar corriendo en http://localhost:8000.")
    print("    Por favor inicie el backend ejecutando: python run.py o start_project.bat")

---
## 👤 HU-01: Registro de Nuevos Clientes (RF-027)
**Estimación:** 5 Story Points | **Prioridad:** Must Have | **Responsable:** Edwin Osorio  
Demuestra el autoservicio de clientes con DNI único, correo válido y encriptación de contraseña con `bcrypt`.

In [ ]:
# Generamos un DNI aleatorio para garantizar reproducibilidad e idempotencia
random_dni = str(random.randint(1000000000, 9999999999))
new_client_payload = {
    "dni": random_dni,
    "email": f"lector_{random_dni[-4:]}@utp.edu.co",
    "nombre": "Esteban",
    "apellido": "Duque",
    "telefono": "3154443322",
    "direccion": "Calle 14 # 23-05, Pereira",
    "password": "LectorSeguro2026!"
}

res_reg = requests.post(f"{BASE_URL}/auth/register", json=new_client_payload)
print(f"Estado HTTP: {res_reg.status_code}")
print("Respuesta del API (Cliente Registrado):")
print(json.dumps(res_reg.json(), indent=2, ensure_ascii=False))
assert res_reg.status_code == 201
print("\n[✓] HU-01 APROBADA: Cliente creado con rol CLIENTE y contraseña protegida por bcrypt.")

---
## 🔐 HU-04: Autenticación y Emisión de Token JWT (RF-026 / RF-028)
**Estimación:** 3 Story Points | **Prioridad:** Must Have | **Responsable:** Jose David Marín  
Demuestra el inicio de sesión y la generación del token Bearer JWT con claims tipados.

In [ ]:
# 1. Login con el nuevo cliente registrado
login_payload = {
    "email": new_client_payload["email"],
    "password": new_client_payload["password"]
}

res_login = requests.post(f"{BASE_URL}/auth/login", json=login_payload)
login_data = res_login.json()
client_token = login_data["access_token"]

print(f"Estado HTTP: {res_login.status_code}")
print(f"Usuario Autenticado: {login_data.get('nombre')} {login_data.get('apellido')}")
print(f"Rol Asignado: {login_data.get('rol')}")
print(f"Token Bearer (primeros 50 caracteres): {client_token[:50]}...")

assert res_login.status_code == 200
assert login_data["rol"] == "CLIENTE"
print("\n[✓] HU-04 APROBADA: Autenticación JWT exitosa y claims verificados.")

---
## 👤 HU-07: Consulta y Actualización de Perfil (RF-021 / RF-028)
**Estimación:** 3 Story Points | **Prioridad:** Must Have | **Responsable:** Jose David Marín  
Demuestra la consulta protegida del perfil (`GET /me`) y la edición de teléfono y dirección (`PUT /me`).

In [ ]:
headers = {"Authorization": f"Bearer {client_token}"}

# 1. Consultar perfil actual
res_me = requests.get(f"{BASE_URL}/auth/me", headers=headers)
print("Perfil Actual:")
print(json.dumps(res_me.json(), indent=2, ensure_ascii=False))
assert res_me.status_code == 200

# 2. Actualizar teléfono y dirección de correspondencia
update_payload = {
    "telefono": "3209998877",
    "direccion": "Avenida Circunvalar # 10-25, Pereira"
}
res_update = requests.put(f"{BASE_URL}/auth/me", headers=headers, json=update_payload)
print("\nPerfil Actualizado:")
print(json.dumps(res_update.json(), indent=2, ensure_ascii=False))
assert res_update.status_code == 200
assert res_update.json()["telefono"] == "3209998877"
print("\n[✓] HU-07 APROBADA: Consulta y actualización de perfil realizada con éxito.")

---
## 🔑 HU-06: Cambio Seguro de Contraseña (RF-028)
**Estimación:** 2 Story Points | **Prioridad:** Must Have | **Responsable:** Edwin Osorio  
Demuestra la validación de la contraseña anterior antes de permitir actualizar la clave.

In [ ]:
# 1. Intento con contraseña actual incorrecta (Debe fallar con HTTP 400)
bad_pwd_payload = {
    "current_password": "ClaveEquivocada123!",
    "new_password": "NuevaClaveSegura2026!"
}
res_bad_pwd = requests.put(f"{BASE_URL}/auth/change-password", headers=headers, json=bad_pwd_payload)
print(f"Intento con clave incorrecta: Estado {res_bad_pwd.status_code} - {res_bad_pwd.json().get('detail')}")
assert res_bad_pwd.status_code == 400

# 2. Cambio exitoso con la contraseña actual correcta
ok_pwd_payload = {
    "current_password": new_client_payload["password"],
    "new_password": "NuevaClaveSegura2026!"
}
res_ok_pwd = requests.put(f"{BASE_URL}/auth/change-password", headers=headers, json=ok_pwd_payload)
print(f"Cambio exitoso: Estado {res_ok_pwd.status_code} - {res_ok_pwd.json().get('mensaje')}")
assert res_ok_pwd.status_code == 200
print("\n[✓] HU-06 APROBADA: Regla de verificación de clave previa y rehasheo bcrypt validada.")

---
## 🛡️ HU-02: Creación de Administradores por Superadmin Root (RF-030)
**Estimación:** 5 Story Points | **Prioridad:** Must Have | **Responsable:** Maicol Londoño  
Demuestra el control RBAC de nivel Superadmin Root para crear personal administrativo y el rechazo a clientes no autorizados.

In [ ]:
# 1. Login como Superadmin Root precargado
res_root_login = requests.post(f"{BASE_URL}/auth/login", json={
    "email": "root@libreria.com",
    "password": "RootSecure2026!"
})
root_token = res_root_login.json()["access_token"]
root_headers = {"Authorization": f"Bearer {root_token}"}

# 2. Intento de un Cliente normal de crear un Administrador (Debe ser bloqueado con HTTP 403 Forbidden)
admin_dni = str(random.randint(10000000, 99999999))
admin_payload = {
    "dni": admin_dni,
    "email": f"admin_demo_{admin_dni[-3:]}@libreria.com",
    "nombre": "Santiago",
    "apellido": "Restrepo",
    "telefono": "3115556677",
    "password": "AdminSecure2026!"
}
res_hacker = requests.post(f"{BASE_URL}/admin/create-admin", headers=headers, json=admin_payload)
print(f"Intento no autorizado de un cliente: Estado {res_hacker.status_code} - Bloqueado por RBAC.")
assert res_hacker.status_code == 403

# 3. Superadmin Root crea legalmente al Administrador
res_admin_created = requests.post(f"{BASE_URL}/admin/create-admin", headers=root_headers, json=admin_payload)
print(f"Creación por Root: Estado {res_admin_created.status_code}")
print("Administrador Creado:")
print(json.dumps(res_admin_created.json(), indent=2, ensure_ascii=False))
assert res_admin_created.status_code == 201
assert res_admin_created.json()["rol"] == "ADMINISTRADOR"
print("\n[✓] HU-02 APROBADA: Control RBAC de creación exclusiva de administradores por Root verificado.")

---
## ⚖️ HU-03: Restricción de Compras para Personal Operativo (RF-031)
**Estimación:** 2 Story Points | **Prioridad:** Must Have | **Responsable:** Maicol Londoño  
Demuestra la aplicación de la regla de oro del proyecto: Administradores y Root tienen terminantemente prohibido comprar o reservar libros en la plataforma.

In [ ]:
# Autenticamos al administrador recién creado
res_adm_login = requests.post(f"{BASE_URL}/auth/login", json={
    "email": admin_payload["email"],
    "password": admin_payload["password"]
})
admin_token = res_adm_login.json()["access_token"]
adm_headers = {"Authorization": f"Bearer {admin_token}"}

intent_payload = {
    "libro_id": 1,
    "cantidad": 1,
    "modalidad": "COMPRA"
}

# 1. Administrador intenta comprar -> DEBE BLOQUEAR CON 403
res_adm_buy = requests.post(f"{BASE_URL}/compras/validar-permiso-compra-reserva", headers=adm_headers, json=intent_payload)
print(f"1. Administrador intenta comprar: HTTP {res_adm_buy.status_code}")
print(f"   Detalle del error: {res_adm_buy.json().get('detail')}")
assert res_adm_buy.status_code == 403

# 2. Superadmin Root intenta reservar -> DEBE BLOQUEAR CON 403
res_root_buy = requests.post(f"{BASE_URL}/compras/validar-permiso-compra-reserva", headers=root_headers, json=intent_payload)
print(f"\n2. Superadmin Root intenta reservar: HTTP {res_root_buy.status_code}")
print(f"   Detalle del error: {res_root_buy.json().get('detail')}")
assert res_root_buy.status_code == 403

# 3. Cliente normal compra -> DEBE PERMITIR CON 200 OK
res_cli_buy = requests.post(f"{BASE_URL}/compras/validar-permiso-compra-reserva", headers=headers, json=intent_payload)
print(f"\n3. Cliente normal intenta comprar: HTTP {res_cli_buy.status_code}")
print(f"   Respuesta: {res_cli_buy.json().get('mensaje')}")
assert res_cli_buy.status_code == 200

print("\n[✓] HU-03 APROBADA: Regla RF-031 aplicada atómicamente por middleware RBAC.")

---
## 🚪 HU-05: Cierre de Sesión (Logout)
**Estimación:** 1 Story Point | **Prioridad:** Must Have | **Responsable:** Jose David Marín  
Demuestra la notificación de cierre de sesión e invalidación de token.

In [ ]:
res_logout = requests.post(f"{BASE_URL}/auth/logout", headers=headers)
print(f"Estado HTTP: {res_logout.status_code}")
print("Respuesta del API:")
print(json.dumps(res_logout.json(), indent=2, ensure_ascii=False))
assert res_logout.status_code == 200
print("\n[✓] HU-05 APROBADA: Sesión finalizada correctamente.")

---
## 📊 Resumen Ejecutivo de Cumplimiento del Sprint 1

| ID Historia | Título de la Historia | Estimación | Responsable | Estado | Criterio de Aceptación Cumplido |
| :---: | :--- | :---: | :--- | :---: | :---: |
| **HU-01** | Registro de nuevos clientes | 5 SP | Edwin Osorio | **DONE** | DNI y correo único, contraseña hasheada con bcrypt |
| **HU-04** | Inicio de sesión con JWT | 3 SP | Jose David Marín | **DONE** | Token Bearer JWT emitido con claims tipados |
| **HU-05** | Cierre de sesión (Logout) | 1 SP | Jose David Marín | **DONE** | Invalidación confirmada en cliente y backend |
| **HU-06** | Cambio seguro de contraseña | 2 SP | Edwin Osorio | **DONE** | Validación obligatoria de contraseña previa |
| **HU-07** | Perfil y actualización de datos | 3 SP | Jose David Marín | **DONE** | Actualización de teléfono y dirección en BD |
| **HU-02** | Creación de Admin por Root | 5 SP | Maicol Londoño | **DONE** | Exclusividad de Root y bloqueo 403 a clientes |
| **HU-03** | Restricción de Compra (RF-031) | 2 SP | Maicol Londoño | **DONE** | Bloqueo 403 a Admin/Root y 200 OK a Cliente |

**Total Story Points Completados:** **21 / 21 SP (100% de cumplimiento)**  
**Definición de Terminado (DoD):** Totalmente integrada en backend FastAPI y frontend React, probada con assertions y ejecutable de principio a fin.